In [ ]:
from pathlib import Path

import torch
import numpy as np
from torch.amp import GradScaler
import time
from collections import Counter
import matplotlib.pyplot as plt

from src.utils.config import load_experiment_config
from src.utils.labels import save_label_names
from src.utils.paths import build_fold_checkpoint_path, ensure_output_directory
from src.utils.runtime import resolve_device
from src.transforms.classification import build_finetune_transform, build_train_transform, build_validation_transform
from src.datasets.fruit_freshness import FruitHFDataset, load_fruit_freshness_dataset
from src.datasets.folds import iter_stratified_folds, select_fold_datasets
from src.datasets.loaders import build_fold_dataloaders, build_holdout_dataloader
from src.models.factory import build_cmt_classifier
from src.losses.focal import FocalLoss, build_class_balanced_alpha
from src.engine.checkpoint import save_model_state
from src.engine.ema import ModelEma
from src.engine.optimization import build_optimizer, build_scheduler
from src.trainers.loops import train_one_epoch, validate_one_epoch
from src.evaluation.metrics import compute_validation_metrics
from src.inference.loading import load_fold_models
from src.inference.ensemble import run_ensemble_holdout

CONFIG_PATH = Path("configs/deep3.toml")


In [ ]:
device = resolve_device()  # 사용 디바이스 선택(CUDA 우선)

train_transform = build_train_transform()
val_transform = build_validation_transform()




In [ ]:
# 메인
def main():  # 전체 파이프라인 실행 함수
    device = resolve_device()
    print("device:", device)
    if torch.cuda.is_available():
        print(torch.cuda.get_device_name(0))

    config = load_experiment_config(CONFIG_PATH)

    final_dataset = load_fruit_freshness_dataset()
    names = final_dataset["train"].features["label"].names
    save_dir = ensure_output_directory("C:/Users/user/Desktop/deep/model_data")
    save_label_names(names, save_dir)

    num_classes = len(final_dataset["train"].features["label"].names)

    # ----- class-balanced alpha (FocalLoss용) -----
    train_labels = [int(x) for x in final_dataset["train"]["label"]]
    counts = Counter(train_labels)
    class_counts = [counts[i] for i in range(num_classes)]
    beta = config["loss"]["class_balanced_beta"]
    alpha = build_class_balanced_alpha(class_counts, beta, num_classes)
    print("alpha:", alpha.tolist())


    # ================= [설정 변경] =================
    EPOCHS = config["training"]["epochs"]
    # 마지막 5 epoch는 파인튜닝 (Cool-down)
    FINETUNE_EPOCHS = config["fine_tuning"]["epochs"]
    
    BATCH_SIZE = config["training"]["batch_size"]
    K = config["cross_validation"]["n_splits"]
    
    # Mixup 설정
    MIXUP_ALPHA = config["mixup"]["alpha"]  # Mixup 강도 (0.08 -> 0.8로 약간 높임, 데이터가 적으면 강한게 좋음)
    MIXUP_P = config["mixup"]["probability"]
    
    # 학습률
    LR_CNN = config["optimization"]["lr_cnn"]
    LR_TRANS = config["optimization"]["lr_trans"]
    WEIGHT_DECAY = config["optimization"]["weight_decay"]
    
    # [전략] EMA Decay 설정
    EMA_DECAY = config["ema"]["decay"]  
    
    USE_CE_LS = config["loss"]["use_ce_label_smoothing"]
    LABEL_SMOOTHING = config["loss"]["label_smoothing"]

    # [전략] 파인튜닝용 약한 증강 (ResizeCrop + Flip만 하고, ColorJitter/Erasing 제거)
    ft_transform = build_finetune_transform()
    # ===============================================


    fold_accs = []
    start_time = time.time()
    histories = []
    
    for fold, (train_idx, val_idx) in enumerate(iter_stratified_folds(final_dataset["train"], n_splits=K, shuffle=config["cross_validation"]["shuffle"], random_state=config["cross_validation"]["random_state"]), 1):
        best_acc_fold = 0.0
        print(f"\n================ Fold {fold}/{K} 시작 ================")
        fold_start = time.time()
            
        history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

        train_split, val_split = select_fold_datasets(final_dataset["train"], train_idx, val_idx)

        # 데이터셋 생성
        train_ds = FruitHFDataset(train_split, transform=train_transform)
        val_ds   = FruitHFDataset(val_split,  transform=val_transform)

        train_loader, val_loader = build_fold_dataloaders(train_ds, val_ds, BATCH_SIZE)
        
        torch.backends.cudnn.benchmark = config["runtime"]["cudnn_benchmark"]

        # --- 모델 초기화 ---
        model = build_cmt_classifier(num_classes).to(device)
        
        # [전략] EMA 모델 초기화
        ema = ModelEma(model, decay=EMA_DECAY, device=device)

        if USE_CE_LS:
            criterion = torch.nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING).to(device)
        else:
            criterion = FocalLoss(alpha=alpha.to(device), gamma=config["loss"]["focal_gamma"]).to(device)

        optimizer = build_optimizer(
            model,
            lr_cnn=LR_CNN,
            lr_trans=LR_TRANS,
            weight_decay=WEIGHT_DECAY,
        )
        scheduler = build_scheduler(optimizer, t_max=EPOCHS)
        scaler = GradScaler()

        val_acc_list = []
        val_loss_list = []
        val_f1_list   = []

        for epoch in range(1, EPOCHS+1):
            epoch_start = time.time()
            
            # ================= [전략 1] 파인튜닝 (Cool-down) 체크 =================
            is_finetuning = (epoch > EPOCHS - FINETUNE_EPOCHS)
            
            if is_finetuning:
                print(f"▶ Fold {fold} | Epoch {epoch} [Fine-tuning Mode! Mixup OFF, Weak Aug]")
                # 1. 증강 교체 (강한 증강 -> 약한 증강)
                train_ds.tf = ft_transform 
                # 2. Mixup OFF (아래 루프에서 처리)
            else:
                print(f"\n▶ Fold {fold} | Epoch {epoch}/{EPOCHS}")
            # ======================================================================

            # ---- [Train] ----
            tr_acc, tr_loss = train_one_epoch(
                model,
                train_loader,
                criterion,
                optimizer,
                device,
                scaler,
                ema,
                is_finetuning,
                MIXUP_P,
                MIXUP_ALPHA,
                progress_description=f"Fold {fold} Epoch {epoch} [Train]",
            )

            # ---- [Validation] ----
            # [전략] 검증 시 EMA 모델 사용 (성능이 더 안정적임)
            # 주의: EMA 모델은 eval 모드이므로 model.eval() 불필요하지만 명시적으로 둠
            val_model = ema.module 
            va_acc, va_loss, all_preds, all_labels, all_logits = validate_one_epoch(
                val_model,
                val_loader,
                criterion,
                device,
                progress_description=f"Fold {fold} Epoch {epoch} [Val]",
            )

            history["train_loss"].append(tr_loss)
            history["train_acc"].append(tr_acc)
            history["val_loss"].append(va_loss)
            history["val_acc"].append(va_acc)
            
            all_logits = np.concatenate(all_logits, axis=0)
            va_f1, va_bal, va_top2, va_top3 = compute_validation_metrics(
                all_labels, all_preds, all_logits
            )

            val_acc_list.append(va_acc)
            val_loss_list.append(va_loss)
            val_f1_list.append(va_f1)
            print(f"Val (EMA) ▶ acc: {va_acc:.4f} | f1: {va_f1:.4f} | loss: {va_loss:.4f}")
            
            # 저장
            if va_acc > best_acc_fold + 1e-6:
                best_acc_fold = va_acc
                save_path = build_fold_checkpoint_path(save_dir, fold)
                # [전략] EMA 모델의 가중치를 저장
                save_model_state(ema.module, save_path)
                print(f"New best model (EMA) saved! (fold={fold}, acc={best_acc_fold:.4f})")

            epoch_time = time.time() - epoch_start
            print(f"Epoch {epoch} 완료 (소요시간: {epoch_time:.2f}초)")

            scheduler.step()

        # --- Fold 종료 ---
        fold_time = time.time() - fold_start
        histories.append(history)
        print(f"✅ Fold {fold} 완료! (소요시간: {fold_time/60:.2f}분)")
        fold_accs.append(val_acc_list)

    # --- 전체 요약 ---
    ckpt_dir = save_dir
    models = load_fold_models(K, num_classes, device, ckpt_dir)

    test_ds = FruitHFDataset(final_dataset["test"], transform=val_transform)
    test_loader = build_holdout_dataloader(test_ds, BATCH_SIZE)

    print("\n[최종 평가] Holdout Test Set (Ensemble + TTA)")
    
    # [전략] TTA + 앙상블 적용
    t_correct, t_total = run_ensemble_holdout(models, test_loader, device)

    print("Final Holdout Acc:", t_correct / t_total)
    
    total_time = time.time() - start_time
    print(f"\n================ 학습종료 총 소요시간: {total_time/60:.2f}분 ================")

    # 그래프 그리기
    epochs = range(1, EPOCHS + 1)
    plt.figure(figsize=tuple(config["reporting"]["figure_size"]))
    plt.subplot(1, 2, 1)
    # 마지막 Fold의 history만 그림
    plt.plot(epochs, history["train_loss"], label="train_loss")
    plt.plot(epochs, history["val_loss"],   label="val_loss")
    plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.title("Loss (Last Fold)"); plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(epochs, history["train_acc"], label="train_acc")
    plt.plot(epochs, history["val_acc"],   label="val_acc")
    plt.xlabel("Epoch"); plt.ylabel("Acc"); plt.title("Accuracy (Last Fold)"); plt.legend()
    plt.tight_layout()
    plt.show()

    save_model_state(model, config["checkpoint"]["final_model_filename"])

if __name__ == "__main__":
    main()